### Hybrid Retriever - Combining Dense and Sparse Retriever

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.schema.document import Document

In [3]:
import langchain_community
print(langchain_community.__version__)

0.0.38


In [4]:
# Step 1: Sample Documents
docs = [
    Document(page_content="LangChain helps you build applications with LLMs and chat models.", metadata={"source": "doc1"}),
    Document(page_content="FAISS is a library for efficient similarity search and clustering of dense vectors.", metadata={"source": "doc2"}),
    Document(page_content="BM25 is a ranking function used by search engines to estimate the relevance of documents to a given search query.", metadata={"source": "doc3"}),
    Document(page_content="Ensemble retrievers combine multiple retrieval strategies to improve search results.", metadata={"source": "doc4"}),
    Document(page_content="The Eiffel Tower is one of the most famous landmarks in Paris.", metadata={"source": "doc5"}),
    Document(page_content="HuggingFace provides a wide range of pre-trained models for natural language processing tasks.", metadata={"source": "doc6"}),
    Document(page_content="The Great Wall of China is a historic fortification built to protect against invasions.", metadata={"source": "doc7"}),
    Document(page_content="LangChain can be used to develop Agentic AI applications.", metadata={"source": "doc8"}),
    Document(page_content="The Pyramids of Giza are ancient monuments located in Egypt.", metadata={"source": "doc9"}),
    Document(page_content="BM25 is based on the probabilistic retrieval framework and is widely used in information retrieval systems.", metadata={"source": "doc10"})
]

In [5]:
# Step 2: Create Dense Retriever (FAISS)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
dense_vector_store = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vector_store.as_retriever()

c:\RAG\env_langchain_lessthan1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\RAG\env_langchain_lessthan1\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
# Step 3: Create Sparse Retriever (BM25)
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3  # Number of top results to return

In [7]:
# Step 4: Combine Retrievers using EnsembleRetriever
hybrid_retriever = EnsembleRetriever(retrievers=[dense_retriever, sparse_retriever], weights=[0.7, 0.3])

In [8]:
# Step 5: Test the Ensemble Retriever
query = "What is LangChain?"
results = hybrid_retriever.get_relevant_documents(query)
print("Ensemble Retriever Results:")
for idx, doc in enumerate(results):
    print(f"{idx + 1}. {doc.page_content} (Source: {doc.metadata['source']})")

Ensemble Retriever Results:
1. LangChain can be used to develop Agentic AI applications. (Source: doc8)
2. LangChain helps you build applications with LLMs and chat models. (Source: doc1)
3. HuggingFace provides a wide range of pre-trained models for natural language processing tasks. (Source: doc6)
4. BM25 is a ranking function used by search engines to estimate the relevance of documents to a given search query. (Source: doc3)
5. BM25 is based on the probabilistic retrieval framework and is widely used in information retrieval systems. (Source: doc10)
6. The Pyramids of Giza are ancient monuments located in Egypt. (Source: doc9)


c:\RAG\env_langchain_lessthan1\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


In [9]:
# Step 5: Test the Hybrid Retriever
query = "How can I build an application using LLMs?"
results = hybrid_retriever.get_relevant_documents(query)
print("Hybrid Retriever Results:")
for idx, doc in enumerate(results):
    print(f"{idx + 1}. {doc.page_content} (Source: {doc.metadata['source']})")

Hybrid Retriever Results:
1. LangChain helps you build applications with LLMs and chat models. (Source: doc1)
2. LangChain can be used to develop Agentic AI applications. (Source: doc8)
3. BM25 is based on the probabilistic retrieval framework and is widely used in information retrieval systems. (Source: doc10)
4. BM25 is a ranking function used by search engines to estimate the relevance of documents to a given search query. (Source: doc3)


### Create a RAG pipeline with Hybrid Retriever

In [10]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

In [11]:
# Prompt Template for Answering Questions
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

In [12]:
# LLM
llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",  # correct argument in <1.0
    temperature=0.2
)

c:\RAG\env_langchain_lessthan1\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


### Create Stuff Document Chain

In [13]:
# Create Chain to Combine Retrieved Documents and Generate Answer
# Create Stuff Document Chain
documents_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

# Create Full RAG Chain
rag_chain = create_retrieval_chain(
    retriever=hybrid_retriever,
    combine_docs_chain=documents_chain
)

rag_chain


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001CFD1BE2590>), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001CFA025B310>, k=3)], weights=[0.7, 0.3]), config={'run_name': 'retrieve_documents'})
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), config={'run_name': 'format_inputs'})
            | PromptTemplate(input_variables=['context', 'input'], template='\nAnswer the question based on the context below.\n\nContext:\n{context}\n\nQuestion: {input}\n')
            | ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001CFE42EC890>, async_client=<openai.resources

In [14]:
# Ask a question to the RAG Chain
query = {"input": "How can I build an application using LLMs?"}
response = rag_chain.invoke(query)

print("RAG Chain Response:")
print(response["answer"])

print("\n---\n")

print("\n Source Documents Used for Answering the Question:")
for idx, doc in enumerate(response["context"]):
    print(f"{idx + 1}. {doc.page_content} (Source: {doc.metadata['source']})")

RAG Chain Response:
You can build an application using LLMs by using LangChain, which helps you develop applications with LLMs and chat models. LangChain provides the tools and resources necessary to incorporate LLMs into your application development process.

---


 Source Documents Used for Answering the Question:
1. LangChain helps you build applications with LLMs and chat models. (Source: doc1)
2. LangChain can be used to develop Agentic AI applications. (Source: doc8)
3. BM25 is based on the probabilistic retrieval framework and is widely used in information retrieval systems. (Source: doc10)
4. BM25 is a ranking function used by search engines to estimate the relevance of documents to a given search query. (Source: doc3)
